# D2 · Residual diagnostics: four mysteries

**Type**: Interpret · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

Implement three diagnostic tests on the residuals (Durbin-Watson, curvature, Breusch-Pagan) and use them to
diagnose four datasets: one respects the assumptions, the other three violate one each.

## Dataset

`ds.residual_mysteries()` returns a dictionary with four DataFrames `"A"`, `"B"`, `"C"`, `"D"`, each with a feature `x`
and a target `y` (200 rows, in order of acquisition). How they were generated is hidden on purpose. The possible
violations are: `"nonlinearity"`, `"heteroscedasticity"`, `"autocorrelation"`. One of the four is healthy (`"ok"`).

## Educational purpose

Move from "I look at the plot" to "I measure". Every pathology has a dedicated statistic and a different remedy, and in
an automatic flow (a model-checking pipeline) plots are not enough.

## Task

1. Write `residuals_of(df)`: OLS fit of `y` on `x` with `np.linalg.lstsq` and return the pair `(x, e)` with the residuals `e`.
2. Write `durbin_watson(e)` = `Σ (e_t − e_{t−1})² / Σ e_t²`. It is about 2 without autocorrelation, and drops toward 0 with positive autocorrelation.
3. Write `curvature_r2(x, e)`: R² of the regression of the residuals on `[1, x, x²]`. A high value means that the residuals have a curved structure left out of the model.
4. Write `breusch_pagan(x, e)`: regress `e²` on `[1, x]`, take `stat = n · R²` and return `(stat, p_value)` with `p_value = scipy.stats.chi2.sf(stat, df=1)`. A small p-value rejects homoscedasticity.
5. Build the DataFrame `table` with index `A`, `B`, `C`, `D` and columns `dw`, `curv_r2`, `p_bp`.
6. Assign to `diagnosis` a dictionary with the violation of each dataset. Suggested rules: `dw < 1.2` → `"autocorrelation"`; `curv_r2 > 0.10` → `"nonlinearity"`; `p_bp < 0.01` → `"heteroscedasticity"`; otherwise `"ok"`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from scipy import stats
from common.plots import residual_panel

mysteries = ds.residual_mysteries()
{name: d.shape for name, d in mysteries.items()}

## Your work

In [ ]:
def residuals_of(df):
    ...

def durbin_watson(e):
    ...

def curvature_r2(x, e):
    ...

def breusch_pagan(x, e):
    ...

In [ ]:
rows = {}
for name, d in mysteries.items():
    x, e = residuals_of(d)
    rows[name] = {"dw": ..., "curv_r2": ..., "p_bp": ...}
table = pd.DataFrame(rows).T
table

In [ ]:
diagnosis = {"A": ..., "B": ..., "C": ..., "D": ...}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='A generic linear regression with numpy: `beta = np.linalg.lstsq(B, z, rcond=None)[0]`, `R² = 1 - SSR/SST` with `SST = ((z - z.mean())**2).sum()`. For the curvature `B = np.column_stack([np.ones_like(x), x, x**2])` and `z = e`; for Breusch-Pagan `B = np.column_stack([np.ones_like(x), x])` and `z = e**2`.'):
    check(close(durbin_watson(np.array([1.0, -1.0, 1.0, -1.0])), 3.0), "durbin_watson is correct on a known case",
          "durbin_watson([1,-1,1,-1]) must equal 12/4 = 3")
    rng_t = np.random.default_rng(0)
    x_t = rng_t.uniform(-3, 3, 300)
    check(curvature_r2(x_t, x_t ** 2 - (x_t ** 2).mean()) > 0.99, "curvature_r2 recognizes a perfect parabola",
          "regressing on [1, x, x^2] an exact parabola must give R2 = 1")
    x_h = rng_t.uniform(1, 10, 500)
    check(breusch_pagan(x_h, rng_t.normal(size=500) * x_h)[1] < 1e-3, "breusch_pagan recognizes growing variance",
          "with errors whose standard deviation grows with x the p-value must be tiny")
    check(breusch_pagan(x_h, rng_t.normal(size=500))[1] > 0.01, "breusch_pagan does not fire on homoscedastic errors",
          "with errors independent of x the p-value should not be small")
    check(list(table.index) == ["A", "B", "C", "D"] and list(table.columns) == ["dw", "curv_r2", "p_bp"],
          "the table has the required index and columns", "index A-D and columns dw, curv_r2, p_bp")
    check(diagnosis == {"A": "heteroscedasticity", "B": "ok", "C": "nonlinearity", "D": "autocorrelation"},
          "correct diagnosis for all four datasets",
          "look at which statistic is out of range in each row of `table`: at most one should fire")
done()

---
The commented solution is in `solutions/D_interpret/D2_residual_diagnostics_four_mysteries.ipynb`. Open it after you pass the check, or after you have really tried.